# Cross-Architecture Energy Prediction — Data Processing Pipeline

This notebook loads and processes traces from two sources:
- **Lotaru** (7 machines, runtime + memory only, no energy)
- **Augur** (4 gpgnodes, runtime + memory + RAPL energy)

and builds task-level energy attributions from raw RAPL package/DRAM counters.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np

## 2. Configuration

In [2]:
LOTARU_BASE_URL = "https://raw.githubusercontent.com/CRC-FONDA/Lotaru-traces/master/traces"

MACHINE_FOLDER_MAP = {
    "a1":          "asok01",
    "a2":          "asok02",
    "c2":          "c2",
    "local":       "local",
    "localRedCpu": "wallyRedCpu",
    "n1":          "n1",
    "n2":          "n2",
}

WORKFLOWS = ["atacseq", "bacass", "chipseq", "eager", "methylseq"]

AUGUR_BASE_URL = "../augur/experiments"


## 3. Lotaru Data Loading

In [3]:
def load_lotaru():
    frames = []
    for machine_name, folder_name in MACHINE_FOLDER_MAP.items():
        for workflow in WORKFLOWS:
            url = f"{LOTARU_BASE_URL}/{machine_name}/results_{workflow}/execution_report_{folder_name}.csv"
            try:
                df = pd.read_csv(url)
            except Exception as e:
                print(f"SKIP {machine_name}/{workflow}: {e}")
                continue
            df["source_machine"] = machine_name
            df["source_workflow"] = workflow   # FIXED — was folder_name
            frames.append(df)

    return pd.concat(frames, ignore_index=True)


In [4]:
lotaru_df = load_lotaru()
print(lotaru_df["source_machine"].value_counts())
print(lotaru_df["source_workflow"].value_counts())

source_machine
a1             1501
a2             1501
c2             1501
n2             1501
n1             1487
local          1333
localRedCpu    1333
Name: count, dtype: int64
source_workflow
chipseq      3450
atacseq      2422
eager        2262
methylseq    1253
bacass        770
Name: count, dtype: int64


In [ ]:
lotaru_df.to_csv("../Processed_data/Raw/lotaru_traces.csv",index=False)

## 4. Energy Processing — Raw RAPL Counters

Cluster-specific RAPL constants (gu-cluster is dual-socket with `energy_1`/`energy_2`; hu-cluster is single-socket with one `energy` column, different wraparound values). Empty DRAM files fall back to package-only energy. Jitter intervals above a plausible-watt ceiling are nulled.

In [5]:
ENERGY_UNIT_DIVISOR = 1_000_000  # verified against the actual RAPL logging tool's units
RAPL_CONSTRAINTS = {
    "gu-cluster": {"pkg": 65532610987,  "dram": 65532610987},
    "hu-cluster": {"pkg": 262143328850, "dram": 65712999613},
}

PLAUSIBLE_MAX_WATTS = {
    "gu-cluster": 300,
    "hu-cluster": 1000,
}
def load_energy_data(path, max_uj, max_watts):
    df = pd.read_csv(path)
    df = df.sort_values("timestamp").reset_index(drop=True)

    # Detect energy columns: gu-cluster has energy_1 + energy_2 (dual socket),
    # hu-cluster has a single 'energy' column (single socket).
    energy_cols = [c for c in df.columns if c.startswith("energy")]

    total_delta = 0
    for col in energy_cols:
        delta = df[col].diff()
        wrapped = delta < 0
        delta.loc[wrapped] = delta.loc[wrapped] + max_uj
        total_delta = total_delta + delta

    df["interval_j"] = total_delta / ENERGY_UNIT_DIVISOR
    df["time_gap_s"] = df["timestamp"].diff() / 1000
    df["implied_watts"] = df["interval_j"] / df["time_gap_s"]

    suspect = df["implied_watts"] > max_watts
    df.loc[suspect, "interval_j"] = None
    return df


In [6]:
def load_node_energy(node, run, run_dir, cluster, verbose=True):
    consts = RAPL_CONSTRAINTS[cluster]
    max_watts = PLAUSIBLE_MAX_WATTS[cluster]

    pkg = load_energy_data(
        os.path.join(run_dir, "energy", f"run_{run}_{node}_pkg.csv"),
        max_uj=consts["pkg"], max_watts=max_watts,
    )

    # DRAM may be empty or missing on some nodes — fall back to pkg-only if so
    dram_path = os.path.join(run_dir, "energy", f"run_{run}_{node}_dram.csv")
    try:
        dram = load_energy_data(dram_path, max_uj=consts["dram"], max_watts=max_watts)
        if dram.empty or dram["interval_j"].notna().sum() == 0:
            raise ValueError("empty dram")
        merged = pd.merge_asof(
            pkg[["timestamp", "interval_j"]],
            dram[["timestamp", "interval_j"]],
            on="timestamp", direction="nearest", tolerance=50,
            suffixes=("_pkg", "_dram"),
        )
        merged["total_interval_j"] = merged["interval_j_pkg"] + merged["interval_j_dram"]
    except (ValueError, pd.errors.EmptyDataError, FileNotFoundError):
        if verbose:
            print(f"  {node}: DRAM missing/empty — using package energy only")
        merged = pkg[["timestamp", "interval_j"]].rename(columns={"interval_j": "total_interval_j"})

    return merged[["timestamp", "total_interval_j"]]

## 5. Task-Level Energy Attribution

In [7]:
def task_energy(task_row, energy_by_host):
    hostname = task_row["hostname"]
    # Find the energy key that matches this hostname
    # (energy keys like 'c40' or 'gpgnode13'; hostnames like 'hu-worker-c40' or 'gpgnode-13')
    host = None
    for key in energy_by_host:
        if key.replace("-", "") in hostname.replace("-", ""):
            host = key
            break
    if host is None:
        return None

    host_energy = energy_by_host[host]
    window = host_energy[
        (host_energy["timestamp"] >= task_row["start"]) &
        (host_energy["timestamp"] <= task_row["complete"])
    ]
    if window.empty:
        return None
    return round(window["total_interval_j"].sum(), 4)

## 6. Concurrency Feature

Window-based energy includes concurrent tasks on the same node; this counts them per task (scoped within a run) so the model can account for it.

In [8]:
# Check for concurrent tasks on the same node during the long task's window
def check_concurrent_tasks(task_row,trace):
    overlapping= trace[
        (trace["hostname"] == task_row["hostname"]) &
        (trace["start"] < task_row["complete"]) &
        (trace["complete"] > task_row["start"]) &
        (trace["task_id"] != task_row["task_id"])
    ]
    
    return len(overlapping)

## 7. Scale Across Clusters, Workflows, and Runs

Runs the full pipeline over both clusters × 4 workflows × 3 runs, detecting nodes per run dynamically and tagging `source_cluster`.

In [9]:
import os

AUGUR_WORKFLOWS = ["atacseq", "chipseq", "nanoseq", "rnaseq"]
AUGUR_CLUSTERS = ["gu-cluster","hu-cluster"]
AUGUR_RUNS = ["1", "2", "3"]

def get_nodes_for_run(run_dir):
    energy_dir = os.path.join(run_dir, "energy")
    pkg_files = [f for f in os.listdir(energy_dir) if f.endswith("_pkg.csv")]
    return sorted(f.replace("_pkg.csv", "").split("_", 2)[2] for f in pkg_files)

def process_augur_run(workflow, run,cluster):
    run_dir = os.path.join(AUGUR_BASE_URL, cluster,workflow, run)

    # Load + filter trace
    trace = pd.read_csv(os.path.join(run_dir, "trace.csv"))
    trace = trace[trace["status"] == "COMPLETED"].copy()
    trace["start"] = trace["start"].astype(int)
    trace["complete"] = trace["complete"].astype(int)

    # Build energy_by_host for nodes that actually have energy files
    nodes = get_nodes_for_run(run_dir)
    energy_by_host = {}
    for node in nodes:
        try:
            energy_by_host[node] = load_node_energy(node, run,run_dir, cluster,verbose=False)
        except Exception as e:
            print(f"  energy load failed {workflow}/{run}/{node}: {e}")

    # Per-task energy + concurrency (concurrency scoped to THIS run only)
    trace["task_energy_j"] = trace.apply(lambda r: task_energy(r, energy_by_host), axis=1)
    trace["concurrent_task_count"] = trace.apply(lambda r: check_concurrent_tasks(r, trace), axis=1)
    trace["start"]= trace["start"].astype(int)
    trace["complete"]= trace["complete"].astype(int)
    trace["source_workflow"] = workflow
    trace["source_run"] = run
    trace["source_cluster"] = cluster
    for col in ["realtime", "peak_rss", "rchar", "cpus"]:
        trace[col] = pd.to_numeric(trace[col], errors="coerce")

    return trace

# Main loop
all_augur = []
for cluster in AUGUR_CLUSTERS:
    for wf in AUGUR_WORKFLOWS:
        for run in AUGUR_RUNS:
            try:
                df = process_augur_run(wf, run,cluster=cluster)
                all_augur.append(df)
                print(f"OK {wf}/{run}: {len(df)} tasks, {df['task_energy_j'].notna().sum()} with energy")
            except Exception as e:
                print(f"SKIP {wf}/{run}: {e}")

augur_all = pd.concat(all_augur, ignore_index=True)
print(f"\nTotal: {len(augur_all)} tasks across {augur_all['source_workflow'].nunique()} workflows")


OK atacseq/1: 268 tasks, 246 with energy
OK atacseq/2: 268 tasks, 260 with energy
OK atacseq/3: 268 tasks, 248 with energy
OK chipseq/1: 314 tasks, 281 with energy
OK chipseq/2: 314 tasks, 289 with energy
OK chipseq/3: 314 tasks, 295 with energy
OK nanoseq/1: 92 tasks, 89 with energy
OK nanoseq/2: 92 tasks, 86 with energy
OK nanoseq/3: 92 tasks, 91 with energy
OK rnaseq/1: 231 tasks, 228 with energy
OK rnaseq/2: 231 tasks, 225 with energy
OK rnaseq/3: 231 tasks, 222 with energy
OK atacseq/1: 268 tasks, 241 with energy
OK atacseq/2: 268 tasks, 243 with energy
OK atacseq/3: 268 tasks, 239 with energy
OK chipseq/1: 3538 tasks, 3157 with energy
OK chipseq/2: 3538 tasks, 3179 with energy
OK chipseq/3: 3538 tasks, 3152 with energy
OK nanoseq/1: 92 tasks, 86 with energy
OK nanoseq/2: 92 tasks, 87 with energy
OK nanoseq/3: 92 tasks, 86 with energy
OK rnaseq/1: 1269 tasks, 1183 with energy
OK rnaseq/2: 1269 tasks, 1187 with energy
OK rnaseq/3: 1269 tasks, 1178 with energy

Total: 18216 tasks ac

In [10]:
print(f"Total: {len(augur_all)} tasks")
print(augur_all.groupby("source_cluster")["source_workflow"].value_counts())  # if you added source_cluster

Total: 18216 tasks
source_cluster  source_workflow
gu-cluster      chipseq              942
                atacseq              804
                rnaseq               693
                nanoseq              276
hu-cluster      chipseq            10614
                rnaseq              3807
                atacseq              804
                nanoseq              276
Name: count, dtype: int64


In [11]:
print(augur_all.groupby("source_cluster")["task_energy_j"].describe())

                  count          mean           std      min        25%  \
source_cluster                                                            
gu-cluster       2560.0  18810.041256  37882.950243   0.0000  1558.4675   
hu-cluster      14018.0  18276.957842  30849.557890  46.7687  1616.4967   

                       50%           75%          max  
source_cluster                                         
gu-cluster      6927.38900  19753.824925  471888.7077  
hu-cluster      6725.41935  23045.838850  752041.8269  


Saving the data into csv

In [ ]:
directory_name = "../Processed_data"
augur_all.to_csv("../Processed_data/Raw/augur_combined_traces_data.csv",index=False)

## 8. Replicate Aggregation

Tasks move between nodes across replicate runs, so aggregation is keyed by `[source_cluster, source_workflow, process, tag, hostname]` — never blending energy across different nodes.

In [35]:
augur_agg = (
    augur_all.groupby(["source_cluster","source_workflow", "process", "tag", "hostname"])
    .agg(
        runtime_s=("realtime", "median"),
        peak_mem=("peak_rss", "median"),
        energy_j=("task_energy_j", "median"),
        runtime_std=("realtime", "std"),
        energy_std=("task_energy_j", "std"),
        rchar=("rchar", "median"),
        cpus=("cpus", "median"),
        concurrent_task_count=("concurrent_task_count", "median"),
        n_replicates=("realtime", "count"),
    )
    .reset_index()
)
print(augur_agg.shape)
print(augur_agg["n_replicates"].value_counts().sort_index())
print(augur_agg["hostname"].value_counts())

(13848, 14)
n_replicates
1    10005
2     3352
3      480
4        5
5        2
7        2
8        1
9        1
Name: count, dtype: int64
hostname
hu-worker-c44    3091
hu-worker-c45    3039
hu-worker-c40    2927
hu-worker-c42    2798
gpgnode-14        562
gpgnode-16        478
gpgnode-13        406
gpgnode-18        390
gpgnode-15        157
Name: count, dtype: int64


## 9. Hardware Specs Tables

Per-machine specs for Lotaru, gu-cluster, and hu-cluster. Benchmark units are not comparable across sources (verified: Lotaru I/O is IOPS, Augur I/O is MB/s), so benchmark features are normalized per-source at modeling time — raw values kept here for transparency.

In [13]:
lotaru_spec_data = {
    "node" : ["local","a1","a2","n1","n2","c2"],
    "cores": [8,8,8,8,8,8],
    "ram":[16,32,32,16,16,32],
    "cpu_benchmark":[458,223,223,369,468,523],
    "io_read":[437,306,341,481,481,481],
    "io_write":[415,301,336,483,483,483],
    "bench_time":[36,40,39,33,30,28]
}
lotaru_spec_table = pd.DataFrame(lotaru_spec_data)

In [14]:
AUGUR_GU_SPEC_DATA_URL = "../augur/tool/infrastructure-profiler/profiles"
gu_nodes_id = [13,14,15,16,18]
data =[]

for node in gu_nodes_id:
    df = pd.read_csv(f"{AUGUR_GU_SPEC_DATA_URL}/gpgnode-{node}.csv")
    df = df.drop("z7b",axis=1)
    df = df.rename(columns={"sysbench":"cpu_benchmark"})
    df_reordered = df.loc[:,["node","cores","ram","cpu_benchmark","io_read","io_write","bench_time"]]
    data.append(df_reordered)

augur_gu_spec_table = pd.concat(data, ignore_index=True)
augur_gu_spec_table
    

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time
0,gpgnode-13,32,64,305.89,97.6,90.6,275.251000
1,gpgnode-14,32,64,306.01,109.0,90.9,230.926333
2,gpgnode-15,32,64,305.84,109.0,90.8,182.168000
3,gpgnode-16,32,64,306.15,109.0,91.4,197.361000
4,gpgnode-18,32,64,306.29,109.0,91.7,261.806000


In [15]:
AUGUR_HU_SPEC_URL= "../augur/tool/infrastructure-profiler/profiles"
hu_nodes = ["c40","c42","c44","c45"]
hu_data=[]
for node in hu_nodes:
    df = pd.read_csv(f"{AUGUR_HU_SPEC_URL}/hu-{node}.csv")
    df = df.drop("z7b",axis=1)
    df = df.rename(columns={"sysbench":"cpu_benchmark"})
    df_reordered = df.loc[:,["node","cores","ram","cpu_benchmark","io_read","io_write","bench_time"]]
    hu_data.append(df_reordered)

augur_hu_spec_table = pd.concat(hu_data,ignore_index=True)

augur_hu_spec_table

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time
0,hu-c40,32,256,1056.95,1362,952,23.160667
1,hu-c42,32,256,1049.80,1380,972,24.036333
2,hu-c44,32,256,1070.40,421,492,33.091667
3,hu-c45,32,256,1066.66,420,499,34.571667


In [16]:
lotaru_spec_table["source_dataset"] = "lotaru"
augur_gu_spec_table["source_dataset"]= "gu_cluster"
augur_hu_spec_table["source_dataset"] = "hu_cluster"
combined_spec_table = pd.concat([lotaru_spec_table,augur_gu_spec_table,augur_hu_spec_table], ignore_index=True)

In [17]:
combined_spec_table

,node,cores,ram,cpu_benchmark,io_read,io_write,bench_time,source_dataset
0,local,8,16,458.00,437.0,415.0,36.000000,lotaru
1,a1,8,32,223.00,306.0,301.0,40.000000,lotaru
2,a2,8,32,223.00,341.0,336.0,39.000000,lotaru
3,n1,8,16,369.00,481.0,483.0,33.000000,lotaru
4,n2,8,16,468.00,481.0,483.0,30.000000,lotaru
5,c2,8,32,523.00,481.0,483.0,28.000000,lotaru
6,gpgnode-13,32,64,305.89,97.6,90.6,275.251000,gu_cluster
7,gpgnode-14,32,64,306.01,109.0,90.9,230.926333,gu_cluster
8,gpgnode-15,32,64,305.84,109.0,90.8,182.168000,gu_cluster
9,gpgnode-16,32,64,306.15,109.0,91.4,197.361000,gu_cluster


In [37]:
print(augur_all.columns.tolist())
print("Augur_agg")
print(augur_agg.columns.tolist())
print(augur_all.groupby("source_cluster").size())

['task_id', 'hostname', 'native_id', 'process', 'tag', 'name', 'status', 'exit', 'module', 'container', 'cpus', 'time', 'disk', 'memory', 'attempt', 'submit', 'start', 'complete', 'duration', 'realtime', 'queue', '%cpu', '%mem', 'rss', 'vmem', 'peak_rss', 'peak_vmem', 'rchar', 'wchar', 'syscr', 'syscw', 'read_bytes', 'write_bytes', 'vol_ctxt', 'inv_ctxt', 'workdir', 'scratch', 'error_action', 'cpu_model', 'task_energy_j', 'concurrent_task_count', 'source_workflow', 'source_run', 'source_cluster', 'hash']
Augur_agg
['source_cluster', 'source_workflow', 'process', 'tag', 'hostname', 'runtime_s', 'peak_mem', 'energy_j', 'runtime_std', 'energy_std', 'rchar', 'cpus', 'concurrent_task_count', 'n_replicates']
source_cluster
gu-cluster     2715
hu-cluster    15501
dtype: int64


In [39]:
print(combined_spec_table.columns.tolist())
print(combined_spec_table["node"].tolist())
print(lotaru_df.columns.tolist())

['node', 'cores', 'ram', 'cpu_benchmark', 'io_read', 'io_write', 'bench_time', 'source_dataset']
['local', 'a1', 'a2', 'n1', 'n2', 'c2', 'gpgnode-13', 'gpgnode-14', 'gpgnode-15', 'gpgnode-16', 'gpgnode-18', 'hu-c40', 'hu-c42', 'hu-c44', 'hu-c45']
['Label', 'Machine', 'Workflow', 'NumberSequences', 'Task', 'WorkflowInputSize', 'Realtime', '%cpu', 'rss', 'rchar', 'wchar', 'cpus', 'read_bytes', 'write_bytes', 'vmem', 'memory', 'peak_rss', 'TaskInputSize', 'TaskInputSizeUncompressed', 'WorkflowInputUncompressed', 'source_machine', 'source_workflow']


In [51]:
def normalize_node(hostname):
    if "worker" in hostname:               # hu-worker-c40 -> hu-c40
        return f"hu-{hostname.split('-')[-1]}"
    return hostname                        # gpgnode-13 stays as is

augur_agg["node"] = augur_agg["hostname"].apply(normalize_node)

missing = set(augur_agg["node"].unique()) - set(combined_spec_table["node"].unique())
print("Augur nodes with no spec match:", missing)

Augur nodes with no spec match: set()


# Creating a Unified schema by combining the Spec Table and Traces

In [53]:
augur_unified = augur_agg.merge(combined_spec_table, on="node", how="left")
augur_unified["source_dataset"] = "augur_" + augur_unified["source_cluster"].str.replace("-cluster", "", regex=False)


In [61]:
lotaru_unified = lotaru_df.rename(columns={
    "Realtime": "runtime_s",
    "peak_rss": "peak_mem",
    "source_machine": "node",
    "Task": "process",
}).copy()

lotaru_unified["energy_j"] = pd.NA
lotaru_unified["concurrent_task_count"] = pd.NA
lotaru_unified["source_dataset"] = "lotaru"
lotaru_unified = lotaru_unified.merge(combined_spec_table, on="node", how="left")


In [62]:
lotaru_unified = lotaru_unified.rename(columns={"source_dataset_x": "source_dataset"}).drop(columns=["source_dataset_y"])

In [55]:
common_cols = [
    "source_dataset", "node", "source_workflow", "process",
    "runtime_s", "peak_mem", "energy_j",
    "rchar", "cpus", "concurrent_task_count",
    "cores", "ram", "cpu_benchmark", "io_read", "io_write",
]

In [58]:
print("augur_unified has source_dataset:", "source_dataset" in augur_unified.columns)
print("lotaru_unified has source_dataset:", "source_dataset" in lotaru_unified.columns)

augur_unified has source_dataset: True
lotaru_unified has source_dataset: False


In [63]:
unified = pd.concat(
    [augur_unified[common_cols], lotaru_unified[common_cols]],
    ignore_index=True
)

In [ ]:
print(unified.shape)
print(unified.groupby("source_dataset").size())
print(unified.isna().sum())


(24005, 15)
source_dataset
augur_gu     1993
augur_hu    11855
lotaru      10157
dtype: int64
source_dataset               0
node                         0
source_workflow              0
process                      0
runtime_s                    0
peak_mem                     3
energy_j                 11284
rchar                        0
cpus                         0
concurrent_task_count    10157
cores                     1333
ram                       1333
cpu_benchmark             1333
io_read                   1333
io_write                  1333
dtype: int64


,source_dataset,node,source_workflow,process,runtime_s,peak_mem,energy_j,rchar,cpus,concurrent_task_count,cores,ram,cpu_benchmark,io_read,io_write
0,augur_gu,gpgnode-14,atacseq,NFCORE_ATACSEQ:ATACSEQ:CUSTOM_DUMPSOFTWAREVERS...,588.0,4816896.0,66.0825,1.023639e+06,1.0,0.0,32.0,64.0,306.01,109.0,90.9
1,augur_gu,gpgnode-16,atacseq,NFCORE_ATACSEQ:ATACSEQ:CUSTOM_DUMPSOFTWAREVERS...,7671.0,4634624.0,248.8336,1.023578e+06,1.0,0.0,32.0,64.0,306.15,109.0,91.4
2,augur_gu,gpgnode-14,atacseq,NFCORE_ATACSEQ:ATACSEQ:FASTQ_ALIGN_BWA:BAM_SOR...,7500.0,7157760.0,621.2081,5.521655e+08,1.0,1.0,32.0,64.0,306.01,109.0,90.9
3,augur_gu,gpgnode-16,atacseq,NFCORE_ATACSEQ:ATACSEQ:FASTQ_ALIGN_BWA:BAM_SOR...,7000.0,7192576.0,585.876,5.521656e+08,1.0,1.0,32.0,64.0,306.15,109.0,91.4
4,augur_gu,gpgnode-14,atacseq,NFCORE_ATACSEQ:ATACSEQ:FASTQ_ALIGN_BWA:BAM_SOR...,12000.0,7196672.0,1052.5113,9.327798e+08,1.0,1.0,32.0,64.0,306.01,109.0,90.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23999,lotaru,n2,methylseq,bismark_report,1375.0,100122624.0,<NA>,3.545091e+06,1.0,<NA>,8.0,16.0,468.00,481.0,483.0
24000,lotaru,n2,methylseq,bismark_align,8470013.0,79200256.0,<NA>,7.126226e+11,2.0,<NA>,8.0,16.0,468.00,481.0,483.0
24001,lotaru,n2,methylseq,bismark_deduplicate,177.0,3502080.0,<NA>,3.579070e+05,2.0,<NA>,8.0,16.0,468.00,481.0,483.0
24002,lotaru,n2,methylseq,qualimap,2969.0,106037248.0,<NA>,1.342586e+07,2.0,<NA>,8.0,16.0,468.00,481.0,483.0


In [68]:
unified = unified[unified["node"] != "localRedCpu"].copy()

In [ ]:
unified.to_csv("../Processed_data/Unified_dataset/unified_dataset.csv",index=False)

In [70]:
print(unified.shape)
print(unified.groupby("source_dataset").size())
print(unified.isna().sum())

(22672, 15)
source_dataset
augur_gu     1993
augur_hu    11855
lotaru       8824
dtype: int64
source_dataset              0
node                        0
source_workflow             0
process                     0
runtime_s                   0
peak_mem                    3
energy_j                 9951
rchar                       0
cpus                        0
concurrent_task_count    8824
cores                       0
ram                         0
cpu_benchmark               0
io_read                     0
io_write                    0
dtype: int64
